# Compare stance prompt runs

This notebook auto-discovers stance runs for one dataset, merges them with the original labels, and shows a simple per-prompt score.

In [48]:
from pathlib import Path
import json
import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", None)

# Change this to the dataset name used under data_out/<dataset_name>/...
DATASET_NAME = "semeval2016_task6_stance"
RUN_TO_USE = 0
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data_in").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RUNS_DIR = PROJECT_ROOT / "data_out" / DATASET_NAME
GOLD_JSON_DIR = PROJECT_ROOT / DATASET_NAME
GOLD_CSV_PATH = PROJECT_ROOT / "data_in" / f"{DATASET_NAME}.csv"
GOLD_SOURCE = GOLD_CSV_PATH if GOLD_CSV_PATH.exists() else GOLD_JSON_DIR

print("Project root:", PROJECT_ROOT)
print("Runs dir:", RUNS_DIR)
print("Gold source:", GOLD_SOURCE)
print("Run used from each file:", RUN_TO_USE)

Project root: /Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper
Runs dir: /Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper/data_out/semeval2016_task6_stance
Gold source: /Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper/data_in/semeval2016_task6_stance.csv
Run used from each file: 0


In [49]:
if GOLD_SOURCE.is_file():
    gold_df = pd.read_csv(GOLD_SOURCE)
    required_cols = {"content", "query", "stance_label"}
    if not required_cols.issubset(gold_df.columns):
        raise ValueError(f"CSV gold source must contain {required_cols}, got {set(gold_df.columns)}")
    if "id" in gold_df.columns:
        gold_df["doc_id"] = gold_df["id"].astype(str)
    elif "doc_id" in gold_df.columns:
        gold_df["doc_id"] = gold_df["doc_id"].astype(str)
    else:
        gold_df["doc_id"] = gold_df.index.astype(str)
    gold_df = gold_df[["doc_id", "query", "content", "stance_label"]].rename(columns={"stance_label": "original_stance"})
elif GOLD_SOURCE.is_dir():
    gold_rows = []
    for path in sorted(GOLD_SOURCE.glob("*.json")):
        with open(path, encoding="utf-8") as f:
            obj = json.load(f)
        gold_rows.append(
            {
                "doc_id": str(obj.get("_id", path.stem)),
                "query": obj.get("query"),
                "content": obj.get("content"),
                "original_stance": obj.get("stance_label"),
            }
        )
    if not gold_rows:
        raise FileNotFoundError(f"No gold JSON files found in {GOLD_SOURCE}")
    gold_df = pd.DataFrame(gold_rows)
else:
    raise FileNotFoundError(f"Gold source not found: {GOLD_SOURCE}")

gold_df = gold_df.sort_values("doc_id").reset_index(drop=True)
DOC_ID_WIDTH = gold_df["doc_id"].astype(str).str.len().max()
gold_df["doc_id"] = gold_df["doc_id"].astype(str).str.zfill(DOC_ID_WIDTH)
print("Doc id width:", DOC_ID_WIDTH)
print("Gold rows:", len(gold_df))
gold_df.head()

Doc id width: 4
Gold rows: 2814


,doc_id,query,content,original_stance
0,1000,Climate Change is a Real Concern,until newyork london frankfurt knee deep in sea #water NOTHING will happen @unfccc #cop21 #co95=FREE parties @annaborgeryd #SemST,pro
1,1001,Climate Change is a Real Concern,".@ClimatParis2015, the climate science forum begins today. Follow its progress on #CFCC15 #SustainableDevelopment #SemST",pro
2,1002,Climate Change is a Real Concern,We need integrated #science with #indigenous knowledge to understand & adapt to #CFCC15 #SemST,pro
3,1003,Climate Change is a Real Concern,".@whelan60 ""While this debate goes on, yet more time is wasted."" #thedrum #SemST",neutral
4,1004,Climate Change is a Real Concern,If sea levels get any higher flooding in lowlands could become dire. #SemST,pro


In [50]:
run_files = sorted(RUNS_DIR.glob("*/document_assignments.csv"))
if not run_files:
    raise FileNotFoundError(f"No stance run files found in {RUNS_DIR}")

used_prompt_names = set()
prompt_runs = []
for assignments_path in run_files:
    run_dir = assignments_path.parent
    summary_path = run_dir / "summary.json"
    if not summary_path.exists():
        continue
    with open(summary_path, encoding="utf-8") as f:
        summary = json.load(f)
    if summary.get("configuration", {}).get("task_type") != "stance_detection":
        continue
    prompt_name = summary.get("configuration", {}).get("stance_prompt_name", run_dir.name)
    if prompt_name in used_prompt_names:
        prompt_name = f"{prompt_name}__{run_dir.name}"
    used_prompt_names.add(prompt_name)

    df = pd.read_csv(assignments_path)
    required_cols = {"doc_id", "predicted_stance"}
    if not required_cols.issubset(df.columns):
        continue
    if "run" in df.columns:
        df = df[df["run"] == RUN_TO_USE].copy()
    df["doc_id"] = df["doc_id"].astype(str).str.zfill(DOC_ID_WIDTH)
    df = df[["doc_id", "predicted_stance"]].rename(columns={"predicted_stance": prompt_name})
    prompt_runs.append((prompt_name, df))

if not prompt_runs:
    raise FileNotFoundError(f"No compatible stance run files found in {RUNS_DIR}")

sorted(name for name, _ in prompt_runs)

['task_definition',
 'task_definition__GenAIStanceOneShot_100_semeval2016_task6_stance_gpt-4o_equal_task_definition']

In [51]:
prompt_columns = [name for name, _ in prompt_runs]

# Compare only the documents actually sampled in the selected run.
sampled_doc_ids = set(prompt_runs[0][1]["doc_id"])
for _, df in prompt_runs[1:]:
    sampled_doc_ids &= set(df["doc_id"])

if not sampled_doc_ids:
    raise ValueError("No overlapping sampled doc_ids found across prompt runs")

comparison_df = gold_df[gold_df["doc_id"].isin(sampled_doc_ids)].copy()
for prompt_name, df in prompt_runs:
    comparison_df = comparison_df.merge(df, on="doc_id", how="left")

comparison_df = comparison_df[["doc_id", "query", "original_stance", *prompt_columns, "content"]]
comparison_df = comparison_df.sort_values("doc_id").reset_index(drop=True)
print("Comparing sampled docs:", len(comparison_df))
comparison_df.head()


Comparing sampled docs: 100


,doc_id,query,original_stance,task_definition,task_definition__GenAIStanceOneShot_100_semeval2016_task6_stance_gpt-4o_equal_task_definition,content
0,0137,Atheism,neutral,neutral,neutral,You better go get my ball.... #SemST
1,0139,Atheism,against,against,against,"#Life Is The Question, #Islam Is The Answe #islam #allah #quran #SemST"
2,0160,Atheism,pro,against,pro,"Dear believers, To prove the book you must NOT read from the book. Regards, an #SemST"
3,0220,Atheism,pro,against,pro,"That nagging doubt you keep having about god? Gods not testing u, it's your intellect trying to tell you your beliefs are bullshit. #SemST"
4,0244,Atheism,pro,against,pro,"Sikhism is not any better either, we have to finish all religions. ;-) @Saimarani13 @BeingHu62727983 @MaheshHindu @Po_st @Swamy39 #SemST"


In [52]:
score_rows = []
total = len(comparison_df)
for prompt_name in prompt_columns:
    correct = (comparison_df[prompt_name] == comparison_df["original_stance"]).sum()
    score_rows.append(
        {
            "prompt_name": prompt_name,
            "correct": int(correct),
            "total": int(total),
            "score": f"{correct}/{total}",
            "accuracy": round(correct / total, 3) if total else None,
        }
    )

scores_df = pd.DataFrame(score_rows).sort_values(["correct", "prompt_name"], ascending=[False, True]).reset_index(drop=True)
scores_df

,prompt_name,correct,total,score,accuracy
0,task_definition__GenAIStanceOneShot_100_semeval2016_task6_stance_gpt-4o_equal_task_definition,76,100,76/100,0.76
1,task_definition,60,100,60/100,0.60


In [53]:
# Optional: only rows where at least one prompt disagrees with the gold label.
mismatch_mask = pd.Series(False, index=comparison_df.index)
for prompt_name in prompt_columns:
    mismatch_mask = mismatch_mask | (comparison_df[prompt_name] != comparison_df["original_stance"])

comparison_df[mismatch_mask].reset_index(drop=True)

,doc_id,query,original_stance,task_definition,task_definition__GenAIStanceOneShot_100_semeval2016_task6_stance_gpt-4o_equal_task_definition,content
0,0160,Atheism,pro,against,pro,"Dear believers, To prove the book you must NOT read from the book. Regards, an #SemST"
1,0220,Atheism,pro,against,pro,"That nagging doubt you keep having about god? Gods not testing u, it's your intellect trying to tell you your beliefs are bullshit. #SemST"
2,0244,Atheism,pro,against,pro,"Sikhism is not any better either, we have to finish all religions. ;-) @Saimarani13 @BeingHu62727983 @MaheshHindu @Po_st @Swamy39 #SemST"
3,0255,Atheism,against,neutral,neutral,"@skepticpedi I don't trust you performing your ""science-based medicine"" on children. I don't think it's science, or why reiterate? #SemST"
4,0279,Atheism,neutral,pro,neutral,@POTUS sweet! Congratulations to a rational decision. #SemST
5,0297,Atheism,against,neutral,neutral,For the #oppression of the #poor for the sighing of the #needy now will I arise. #Psalms 12:5 #Bible #God #SemST
6,0343,Atheism,against,neutral,neutral,I am strong because I am wise and full of knowledge -Prov. 24:5 #SemST
7,0493,Atheism,against,neutral,neutral,"Take away hatred from some people, and you have men without faith. ~Eric Hoffer #SemST"
8,0542,Atheism,against,neutral,pro,@DrAliceRoberts was awarded Humanist of the Year at the gala dinner and she even signed her book for us #BHA2015 #SemST
9,0559,Atheism,against,neutral,neutral,"Atheists don't believe in Satan either, so all you Satanists out there, put the sacrifice down and go have a beer! #beer #SemST"
